# Graph construction — the compendium as graphs

Builds every graph the GNN experiments read, from artifacts Phases 0–3 already
produced, and asserts each construction property before anything is trained.
Nothing here is written to `reports/`; this notebook exists so a broken graph is
caught here rather than showing up later as a strange AUC.

Two framings:

- **One heterogeneous graph** (G1 / G2 / G3) — sample nodes, taxon leaves and
  internal taxonomy nodes. A sample's composition lives only on its
  sample → taxon edges.
- **One small graph per sample** — the fixed 666-node taxonomy skeleton with that
  sample's CLR values on the leaves. Graphs are disjoint, so no sample can
  influence another's prediction.

## Setup

In [1]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import differential as diff
from src import graphs as gr

INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"
REL_PATH = PROCESSED / "abund_relative.parquet"
CLR_PATH = PROCESSED / "abund_clr.parquet"

with open(ROOT / "config" / "params.yaml") as f:
    params = yaml.safe_load(f)
SEED = params["seed"]
gcfg = params["graph"]

import torch
import torch_geometric
import torch_geometric.typing as pyg_typing
warnings.filterwarnings("ignore")
torch.manual_seed(SEED)

print(f"seed={SEED}  torch={torch.__version__}  torch_geometric={torch_geometric.__version__}  "
      f"pyg_lib={pyg_typing.WITH_PYG_LIB}  cuda={torch.cuda.is_available()}")
print(f"knn_k={gcfg['knn_k']}  variants={gcfg['variants']}")

C:\Users\Jadrovski\vnp\MicroWave\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


seed=42  torch=2.14.0+cpu  torch_geometric=2.8.1  pyg_lib=True  cuda=False
knn_k=10  variants=['G1', 'G2', 'G3']


## 0 — Environment check

A 3-node `HeteroData` through a heterogeneous GraphSAGE layer: if this runs, the
PyG install is usable.

In [2]:
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, to_hetero

toy = HeteroData()
toy["sample"].x = torch.randn(2, 2)
toy["taxon"].x = torch.randn(1, 2)
toy["sample", "abundance", "taxon"].edge_index = torch.tensor([[0, 1], [0, 0]])
toy["taxon", "rev_abundance", "sample"].edge_index = torch.tensor([[0, 0], [0, 1]])

class _Toy(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = SAGEConv((-1, -1), 4)

    def forward(self, x, edge_index):
        return self.conv(x, edge_index)

out = to_hetero(_Toy(), toy.metadata(), aggr="sum")(toy.x_dict, toy.edge_index_dict)
print({k: tuple(v.shape) for k, v in out.items()})
assert out["sample"].shape == (2, 4) and out["taxon"].shape == (1, 4)

{'sample': (2, 4), 'taxon': (1, 4)}


## 1 — Taxonomy skeleton

Nodes are keyed by the full lineage tuple, not by name. The test that matters:
the 44 genera called `Incertae Sedis` and the 46 with genus `NA` must stay 44 and
46 distinct leaves — keyed by name they would collapse to one node each and fuse
unrelated clades.

In [3]:
taxa = [c for c in pq.ParquetFile(CLR_PATH).schema.names if c != gr.SAMPLE_COL]
taxon_table = pd.read_parquet(INTERIM / "taxon_table.parquet")
taxonomy = gr.build_taxonomy_edges(taxon_table, taxa)
tax_edges, node_index, rank_of_node = taxonomy

gr.check_taxonomy(node_index, tax_edges, n_leaves=419, n_internal=247)
assert len(node_index) == 666

leaves = node_index[node_index["is_leaf"]]
for name, expected in (("Incertae Sedis", 44), ("NA", 46)):
    n = int((leaves["name"] == name).sum())
    assert n == expected and leaves.loc[leaves["name"] == name, "lineage"].nunique() == expected
    print(f"genus {name!r}: {n} taxa -> {n} distinct leaf nodes")

roots = node_index[node_index["parent"] < 0]
print(f"{len(node_index)} nodes ({len(leaves)} leaves, {len(node_index) - len(leaves)} internal), "
      f"{tax_edges.shape[1]} child->parent edges, {len(roots)} roots: {list(roots['name'])}")
node_index.groupby("rank_name", sort=False).size().rename("lineage nodes").to_frame().T

genus 'Incertae Sedis': 44 taxa -> 44 distinct leaf nodes
genus 'NA': 46 taxa -> 46 distinct leaf nodes
666 nodes (419 leaves, 247 internal), 662 child->parent edges, 4 roots: ['Archaea', 'Bacteria', 'Eukaryota', 'NA']


rank_name,genus,kingdom,phylum,class,order,family
lineage nodes,419,4,18,27,67,131


## 2 — Abundance edges, full QC-passed population

One edge per non-zero relative abundance over the 419 modelled taxa, for every
QC-passed sample (116,657 — including the one sample with no CLR row, which simply
has no edges). Weight = `log1p(relative × 10⁴)`, non-negative; the signed CLR value
travels as an edge attribute.

In [4]:
t0 = time.time()
all_samples = pq.read_table(REL_PATH, columns=[gr.SAMPLE_COL]).column(0).to_pylist()
full_index, full_weight, full_attr = gr.build_bipartite_edges(REL_PATH, CLR_PATH, all_samples, taxa)
per_sample = gr.edges_per_sample(full_index, len(all_samples))
print(f"{len(all_samples):,} samples, {full_index.shape[1]:,} edges "
      f"({full_index.shape[1] / (len(all_samples) * len(taxa)):.2%} dense)  [{time.time() - t0:.0f}s]")
print(f"edges per sample: median {np.median(per_sample):.0f}, "
      f"p10 {np.percentile(per_sample, 10):.0f}, p90 {np.percentile(per_sample, 90):.0f}, "
      f"zero-edge samples {(per_sample == 0).sum()}")

assert full_index.shape[1] == 6_600_238
assert (full_weight >= 0).all() and torch.equal(full_weight, full_attr[:, 1])
assert (np.median(per_sample), np.percentile(per_sample, 10), np.percentile(per_sample, 90)) == (51, 9, 109)
del full_index, full_weight, full_attr

116,657 samples, 6,600,238 edges (13.50% dense)  [4s]
edges per sample: median 51, p10 9, p90 109, zero-edge samples 1


## 3 — The labelled cohort: everything G1–G3 need

The within-project cohort (14,033 samples, 41 projects) is the population the
disease experiments use. Taxon prevalence / mean CLR are computed over all of it
here only to check construction; the CV experiments recompute them on each fold's
training rows.

In [5]:
cohort = pd.read_parquet(PROCESSED / "cohort_within_project.parquet")
cohort["sample"] = diff.sample_keys(cohort)
samples = cohort["sample"].tolist()
projects = cohort["project"].to_numpy()
y = (cohort["disease_label"] == "case").astype(int).to_numpy()

bipartite = gr.build_bipartite_edges(REL_PATH, CLR_PATH, samples, taxa)
clr_df, _ = diff.load_clr(CLR_PATH, samples)
clr_df = clr_df.loc[samples, taxa]
presence = gr.presence_from_edges(bipartite[0], len(samples), len(taxa))
assert np.allclose(bipartite[2][:, 0].numpy(), clr_df.to_numpy()[bipartite[0][0], bipartite[0][1]])

leaf_stats = gr.taxon_leaf_stats(clr_df.to_numpy(), presence)
sample_x, sample_feature_names = gr.sample_node_features(
    pd.read_parquet(INTERIM / "sample_depth.parquet"), samples)
gr.assert_no_composition(sample_feature_names, taxa)

knn = gr.build_knn_edges(clr_df.to_numpy(), samples, projects, gcfg["knn_k"])
knn_filtered = gr.build_knn_edges(clr_df.to_numpy(), samples, projects, gcfg["knn_k"],
                                  drop_same_project=True)
harmonized = pd.read_parquet(INTERIM / "samples_harmonized.parquet")
project_x, project_feature_names = gr.project_node_features(harmonized, sorted(set(projects)))

per_sample = gr.edges_per_sample(bipartite[0], len(samples))
print(f"{len(samples):,} samples / {len(set(projects))} projects / {y.sum():,} cases")
print(f"abundance edges: {bipartite[0].shape[1]:,} (median {np.median(per_sample):.0f} per sample)")
print(f"sample features: {sample_feature_names}")
print(f"k-NN (k={gcfg['knn_k']}): {knn[0].shape[1]:,} symmetric edges; "
      f"same-project neighbours {knn[4]:.1%}; after deleting them {knn_filtered[0].shape[1]:,} edges remain")
print(f"project features: {project_x.shape[1]} columns over {len(project_feature_names)} covariate levels")

14,033 samples / 41 projects / 5,669 cases
abundance edges: 972,145 (median 68 per sample)
sample features: ['log10_depth', 'n_nonzero']
k-NN (k=10): 247,900 symmetric edges; same-project neighbours 67.7%; after deleting them 87,412 edges remain
project features: 46 columns over 46 covariate levels


## 4 — G1, G2, G3

Same samples, taxa and taxonomy in all three; they differ only in the extra edges.
G1 and G3 must contain no sample–sample edges. The same-project flag on G2's edges
is stored outside `edge_attr`, so it cannot reach a model.

In [6]:
common = dict(projects=projects, taxonomy=taxonomy, bipartite=bipartite, sample_x=sample_x,
              sample_feature_names=sample_feature_names, taxa_cols=taxa,
              leaf_stats=leaf_stats, y=y)
graphs = {
    "G1": gr.build_hetero_graph("G1", samples, **common),
    "G2": gr.build_hetero_graph("G2", samples, **common, knn=knn),
    "G3": gr.build_hetero_graph("G3", samples, **common, project_x=project_x),
}
SIM = ("sample", "similar_to", "sample")
for name, g in graphs.items():
    g.validate(raise_on_error=True)
    assert g["sample"].x.shape[1] == len(gr.SAMPLE_FEATURES)
    assert (SIM in g.edge_types) == (name == "G2")
assert graphs["G2"][SIM].edge_attr.shape[1] == 1  # distance only; same_project kept apart
print(f"G2 pct_same_project_neighbours = {graphs['G2'].pct_same_project_neighbours:.1%}")

pd.DataFrame(
    {name: {**{f"nodes: {t}": g[t].num_nodes for t in g.node_types},
            **{f"edges: {'-'.join(e)}": g[e].num_edges for e in g.edge_types}}
     for name, g in graphs.items()}
).fillna(0).astype(int)

G2 pct_same_project_neighbours = 67.7%


,G1,G2,G3
nodes: sample,14033,14033,14033
nodes: taxon,419,419,419
nodes: taxon_internal,247,247,247
edges: sample-abundance-taxon,972145,972145,972145
edges: taxon-rev_abundance-sample,972145,972145,972145
edges: taxon-child_of-taxon_internal,419,419,419
edges: taxon_internal-parent_of-taxon,419,419,419
edges: taxon_internal-child_of-taxon_internal,243,243,243
edges: taxon_internal-parent_of-taxon_internal,243,243,243
edges: sample-similar_to-sample,0,247900,0


## 5 — One taxonomy graph per sample

14,033 graphs, every one with the identical `edge_index` (the same tensor object),
so only node features differ between samples.

In [7]:
t0 = time.time()
labels = pd.Series(y, index=samples)
per_sample_graphs = gr.build_per_sample_dataset(
    clr_df, taxonomy, labels, presence=presence, log10_prevalence=leaf_stats[0])
print(f"{len(per_sample_graphs):,} graphs  [{time.time() - t0:.1f}s]")

assert len(per_sample_graphs) == 14_033
first = per_sample_graphs[0].edge_index
assert all(g.edge_index is first for g in per_sample_graphs)
assert first.shape[1] == 2 * tax_edges.shape[1] == 1324
leaf_x = torch.stack([g.x[:419] for g in per_sample_graphs[:200]])
assert torch.allclose(leaf_x[:, :, 0], torch.tensor(clr_df.to_numpy()[:200]))
print(per_sample_graphs[0])

from torch_geometric.loader import DataLoader
print(next(iter(DataLoader(per_sample_graphs, batch_size=256, shuffle=False))))

14,033 graphs  [0.8s]
Data(x=[666, 3], edge_index=[2, 1324], rank=[666], is_leaf=[666], num_nodes=666, sample='PRJEB39382_ERR4341355', y=[1])
DataBatch(x=[170496, 3], edge_index=[2, 338944], rank=[170496], is_leaf=[170496], num_nodes=170496, sample=[256], y=[256], batch=[170496], ptr=[257])


## Summary

Every construction property checked above held. Nothing is trained here; the
models and the training loop are checked in `08_gnn_models.ipynb`, and the
experiments read these builders.